In [1]:
# ---- 설치 안내 ----
# Colab이면 한 번만 실행:
!pip -q install ultralytics openpyxl tqdm scipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.9 MB/s eta 0:00:00


In [2]:
# ==============================
# 캔버스 A (실행용)
# warp 이후 베드 이미지 → lettuce-seg 1회 → mask(RLE, JSONL) 저장
# 6x2 슬롯(t1~t6, b1~b6)로 centroid 매칭(과다 검출은 컷)
# scale_map 반영하여 전 상추 지표 계산 → features.xlsx 저장
# ==============================

import os, re, json, time
from glob import glob
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment

# =========================
# 0) CFG (여기만 수정)
# =========================
BED_WARP_FOLDER = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/ROI_WARP/step6_warp_images/보정 후"   # warp된 베드 이미지 폴더(재귀 탐색)
SCALE_CSV       = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step2. lettuce/260128_daily_scale_1st.csv"      # scale_map CSV
SEG_MODEL_PATH  = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/model/results/runs/bed_seg_v1/weights/best.pt"     # lettuce-seg pt
OUT_DIR         = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step2. lettuce"

# 실행 파라미터
N_WORKERS   = 6          # 이미지 로딩 병렬
BATCH_SIZE  = 12         # YOLO 배치
CHUNK_SIZE  = 1000       # 중간 저장 chunk

# YOLO 파라미터
PRED_CONF = 0.05
PRED_IOU  = 0.7

# 슬롯(고정 prior)
GRID_COLS = 6
GRID_ROWS = 2
X_MARGIN_FRAC = 0.08     # 좌우 마진(이미지 폭 비율)
Y_MARGIN_FRAC = 0.12     # 상하 마진(이미지 높이 비율)
MAX_ASSIGN_DIST_FRAC = 0.20  # (이미지 폭 기준) 너무 먼 매칭 컷

# 정면 전용 추가지표 파라미터 (네 260128 코드 기준)
BOTTOM_BAND_RATIO = 0.12
CORE_Q = 90


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# =========================
# 1) 공통 유틸
# =========================
IMG_PATTERNS = ["**/*.jpg", "**/*.JPG", "**/*.jpeg", "**/*.JPEG", "**/*.png", "**/*.PNG"]
def now_hms():
    return time.strftime("%H:%M:%S")

def collect_image_paths(root: str):
    paths = []
    for pat in IMG_PATTERNS:
        paths.extend(glob(os.path.join(root, pat), recursive=True))
    return sorted(set(paths))

def mask_to_rle(mask_u8: np.ndarray):
    """COCO 스타일 RLE(간단형) - Fortran order"""
    pixels = (mask_u8 > 0).astype(np.uint8).flatten(order="F")
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return runs.tolist()

def get_main_contour(mask_u8: np.ndarray):
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None
    return max(contours, key=cv2.contourArea)

def calc_shape_descriptors(contour):
    out = {
        "bbox_w": np.nan,
        "bbox_h": np.nan,
        "perimeter_px": np.nan,
        "circularity": np.nan,
        "solidity": np.nan,
        "concavity": np.nan,
        "curvature": np.nan,
        "roughness": np.nan,
    }
    if contour is None:
        return out

    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)
    x, y, w, h = cv2.boundingRect(contour)

    out["bbox_w"] = float(w)
    out["bbox_h"] = float(h)
    out["perimeter_px"] = float(perimeter)

    if perimeter > 0:
        out["circularity"] = float(4 * np.pi * area / (perimeter ** 2))

    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)
    if hull_area > 0:
        sol = area / hull_area
        out["solidity"] = float(sol)
        out["concavity"] = float(1 - sol)

    pts = contour.squeeze()
    if len(pts) > 10:
        diffs = np.diff(pts, axis=0)
        angles = np.arctan2(diffs[:, 1], diffs[:, 0])
        curv = np.abs(np.diff(angles))
        out["curvature"] = float(np.mean(curv))
        out["roughness"] = float(np.std(curv))

    return out


def compute_bottom_flatness(mask_u8: np.ndarray, contour, band_ratio: float = 0.12):
    if contour is None:
        return np.nan
    x, y, w, h = cv2.boundingRect(contour)
    if w <= 0 or h <= 0:
        return np.nan

    band = max(3, int(h * band_ratio))
    y0 = min(mask_u8.shape[0] - 1, y + h - 1)
    y1 = max(0, y0 - band)

    region = mask_u8[y1:y0 + 1, x : x + w]
    if region.size == 0:
        return np.nan

    max_span = 0
    for r in region:
        xs = np.where(r > 0)[0]
        if xs.size >= 2:
            span = int(xs.max() - xs.min() + 1)
            if span > max_span:
                max_span = span
        elif xs.size == 1:
            max_span = max(max_span, 1)

    return float(max_span / w)


def compute_core_prominence(mask_u8: np.ndarray, q: int = 90):
    if mask_u8 is None or mask_u8.max() == 0:
        return np.nan

    m255 = (mask_u8 > 0).astype(np.uint8) * 255
    dist = cv2.distanceTransform(m255, cv2.DIST_L2, 5)
    vals = dist[dist > 0]
    if vals.size == 0:
        return np.nan

    thr = np.percentile(vals, q)
    core_ratio = float((dist >= thr).sum() / (dist > 0).sum())
    return core_ratio


def masked_brightness_mean(img_bgr: np.ndarray, mask_u8: np.ndarray):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    m = mask_u8 > 0
    if m.sum() == 0:
        return float(gray.mean())
    return float(gray[m].mean())


def masked_blur_score(img_bgr: np.ndarray, mask_u8: np.ndarray):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    m = mask_u8 > 0
    if m.sum() < 50:
        return float(lap.var())
    return float(lap[m].var())


In [4]:
# =========================
# 2) base_key / 날짜 파싱
# =========================
def parse_base_key_from_path(img_path: str):
    return os.path.splitext(os.path.basename(img_path))[0]

def parse_bed_date_cam(base_key: str):
    """bed03_20251226_080317_cam2 같은 형식에서 bed/date/cam 추출"""
    bed = None
    date = None
    cam = None

    m = re.search(r"(bed\d+)", base_key)
    if m:
        bed = m.group(1)

    m = re.search(r"_(\d{8})_", base_key)
    if m:
        date = m.group(1)

    m = re.search(r"_cam(\d+)", base_key)
    if m:
        cam = m.group(1)

    bed_date_clean = None
    if bed and date and cam:
        bed_date_clean = f"{bed}_{date}_cam{cam}"

    return bed, date, cam, bed_date_clean


# =========================
# 3) scale_map
# =========================

def load_scale_map(scale_csv: str):
    if not isinstance(scale_csv, str) or len(scale_csv) == 0 or not os.path.exists(scale_csv):
        print("[INFO] SCALE_CSV 미사용: cm 단위는 NaN으로 저장됩니다.")
        return None

    df = pd.read_csv(scale_csv)

    if "parent_name" not in df.columns:
        if "image_name" in df.columns:
            df["parent_name"] = df["image_name"].astype(str).apply(lambda x: os.path.splitext(os.path.basename(x))[0])
        else:
            raise ValueError("SCALE_CSV에 parent_name 또는 image_name 컬럼이 필요합니다.")

    smap = df.set_index("parent_name").to_dict(orient="index")
    print(f"[INFO] scale_map 로드 완료: {len(smap)} rows")
    return smap


def scale_lookup(scale_map, base_key: str):
    if scale_map is None or base_key not in scale_map:
        return (np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan)

    info = scale_map[base_key]

    mm_per_px = np.nan
    pxmm_x = np.nan
    pxmm_y = np.nan
    cmppx_x = np.nan
    cmppx_y = np.nan
    cyl_ok = np.nan
    cyl_diam_px = np.nan

    if "cyl_ok" in info and pd.notna(info["cyl_ok"]):
        cyl_ok = float(info["cyl_ok"])
    if "cyl_diam_px" in info and pd.notna(info["cyl_diam_px"]):
        cyl_diam_px = float(info["cyl_diam_px"])

    if "cm_per_px_today" in info and pd.notna(info["cm_per_px_today"]):
        cmppx_x = float(info["cm_per_px_today"])
        cmppx_y = float(info["cm_per_px_today"])
        mm_per_px = cmppx_x * 10.0
        if mm_per_px > 0:
            pxmm_x = 1.0 / mm_per_px
            pxmm_y = 1.0 / mm_per_px
        return (mm_per_px, pxmm_x, pxmm_y, cmppx_x, cmppx_y, cyl_ok, cyl_diam_px)

    if ("px_per_mm_x" in info and "px_per_mm_y" in info and pd.notna(info["px_per_mm_x"]) and pd.notna(info["px_per_mm_y"])):
        pxmm_x = float(info["px_per_mm_x"])
        pxmm_y = float(info["px_per_mm_y"])
        cmppx_x = (1.0 / pxmm_x) / 10.0
        cmppx_y = (1.0 / pxmm_y) / 10.0
        mm_per_px = ((cmppx_x + cmppx_y) / 2.0) * 10.0
        return (mm_per_px, pxmm_x, pxmm_y, cmppx_x, cmppx_y, cyl_ok, cyl_diam_px)

    if "mm_per_px" in info and pd.notna(info["mm_per_px"]):
        mm_per_px = float(info["mm_per_px"])
        cmppx_x = mm_per_px / 10.0
        cmppx_y = mm_per_px / 10.0
        if mm_per_px > 0:
            pxmm_x = 1.0 / mm_per_px
            pxmm_y = 1.0 / mm_per_px
        return (mm_per_px, pxmm_x, pxmm_y, cmppx_x, cmppx_y, cyl_ok, cyl_diam_px)

    return (mm_per_px, pxmm_x, pxmm_y, cmppx_x, cmppx_y, cyl_ok, cyl_diam_px)



In [5]:
# =========================
# 4) 슬롯 정의(6x2) + 매칭
# =========================

def build_slot_centers(W: int, H: int, cols: int = 6, rows: int = 2, x_margin_frac: float = 0.08, y_margin_frac: float = 0.12):
    x0 = int(W * x_margin_frac)
    x1 = int(W * (1 - x_margin_frac))
    y0 = int(H * y_margin_frac)
    y1 = int(H * (1 - y_margin_frac))

    xs = np.linspace(x0, x1, cols)
    ys = np.linspace(y0, y1, rows)

    slot_names = []
    slot_centers = []
    for r in range(rows):
        for c in range(cols):
            name = ("t" if r == 0 else "b") + str(c + 1)
            slot_names.append(name)
            slot_centers.append((float(xs[c]), float(ys[r])))

    return slot_names, np.array(slot_centers, dtype=float)


def assign_instances_to_slots(centroids: np.ndarray, slot_centers: np.ndarray, max_dist: float):
    if centroids.size == 0:
        return {}

    n_i = centroids.shape[0]
    n_s = slot_centers.shape[0]

    C = np.zeros((n_i, n_s), dtype=float)
    for i in range(n_i):
        dx = centroids[i, 0] - slot_centers[:, 0]
        dy = centroids[i, 1] - slot_centers[:, 1]
        C[i, :] = np.hypot(dx, dy)

    r, c = linear_sum_assignment(C)

    inst_to_slot = {}
    for i, j in zip(r, c):
        if C[i, j] <= max_dist:
            inst_to_slot[int(i)] = int(j)

    return inst_to_slot


# =========================
# 5) YOLO 추론 결과 → 인스턴스 리스트
# =========================

def yolo_to_instances(result, W: int, H: int):
    masks = result.masks
    boxes = result.boxes
    if masks is None or boxes is None or len(masks) == 0:
        return []

    conf_list = boxes.conf.detach().cpu().numpy().astype(float)
    masks_np = masks.data.detach().cpu().numpy().astype(np.uint8)

    out = []
    for k in range(len(masks_np)):
        mk = cv2.resize(masks_np[k], (W, H), interpolation=cv2.INTER_NEAREST)
        mk_u8 = (mk > 0).astype(np.uint8) * 255
        area = int(cv2.countNonZero(mk_u8))
        if area <= 0:
            continue

        ys, xs = np.where(mk_u8 > 0)
        cx = float(xs.mean())
        cy = float(ys.mean())

        out.append({
            "mask_u8": mk_u8,
            "conf": float(conf_list[k]),
            "centroid": (cx, cy),
            "area_px": area,
            "raw_idx": int(k),
        })

    return out


In [6]:
# =========================
# 6) 배치 result를 그대로 쓰는 1장 처리
# =========================

def process_bed_image_with_result(img_path: str, img_bgr: np.ndarray, result, scale_map, mask_writer, slot_names, slot_centers, max_assign_dist):
    H, W = img_bgr.shape[:2]
    base_key = parse_base_key_from_path(img_path)

    bed, date, cam, bed_date_clean = parse_bed_date_cam(base_key)
    mm_per_px, pxmm_x, pxmm_y, cmppx_x, cmppx_y, cyl_ok, cyl_diam_px = scale_lookup(scale_map, base_key)
    px_per_mm = float(np.nanmean([pxmm_x, pxmm_y])) if (np.isfinite(pxmm_x) or np.isfinite(pxmm_y)) else np.nan

    instances = yolo_to_instances(result, W=W, H=H)
    n_instances = int(len(instances))

    if n_instances == 0:
        return [{
            "image_path": img_path,
            "base_key": base_key,
            "lettuce_id": np.nan,
            "bed_date": base_key,
            "n_instances": 0,
            "conf": np.nan,
            "brightness_mean": float(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).mean()),
            "blur_score": float(cv2.Laplacian(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()),
            "area_px": 0,
            "area_cm2": np.nan,
            "px_per_mm_x": pxmm_x,
            "px_per_mm_y": pxmm_y,
            "mm_per_px": mm_per_px,
            "cyl_ok": cyl_ok,
            "cyl_diam_px": cyl_diam_px,
            "front_height_cm": np.nan,
            "area_front": 0,
            "aspect_ratio": np.nan,
            "bottom_flatness": np.nan,
            "core_prominence": np.nan,
            "bbox_w": np.nan,
            "bbox_h": np.nan,
            "perimeter_px": np.nan,
            "circularity": np.nan,
            "solidity": np.nan,
            "concavity": np.nan,
            "curvature": np.nan,
            "roughness": np.nan,
            "best_instance": np.nan,
            "position_group": np.nan,
            "bed_date_clean": bed_date_clean,
            "date": date,
            "px_per_mm": px_per_mm,
        }]

    centroids = np.array([inst["centroid"] for inst in instances], dtype=float)
    inst_to_slot = assign_instances_to_slots(centroids, slot_centers, max_dist=max_assign_dist)

    rows = []
    for inst_idx, slot_idx in inst_to_slot.items():
        inst = instances[inst_idx]
        mask_u8 = inst["mask_u8"]
        conf = inst["conf"]

        lettuce_id = slot_names[slot_idx]
        position_group = "t" if lettuce_id.startswith("t") else "b"

        area_px = int(inst["area_px"])
        contour = get_main_contour(mask_u8)
        shape = calc_shape_descriptors(contour)

        if np.isfinite(cmppx_x) and np.isfinite(cmppx_y):
            area_cm2 = float(area_px * (cmppx_x * cmppx_y))
        else:
            area_cm2 = np.nan

        bbox_w = shape.get("bbox_w", np.nan)
        bbox_h = shape.get("bbox_h", np.nan)

        front_height_cm = float(bbox_h * cmppx_y) if (np.isfinite(bbox_h) and np.isfinite(cmppx_y)) else np.nan
        area_front = area_px
        aspect_ratio = float(bbox_w / bbox_h) if (np.isfinite(bbox_w) and np.isfinite(bbox_h) and bbox_h > 0) else np.nan

        bottom_flat = compute_bottom_flatness(mask_u8, contour, band_ratio=BOTTOM_BAND_RATIO)
        core_prom = compute_core_prominence(mask_u8, q=CORE_Q)

        b_mean = masked_brightness_mean(img_bgr, mask_u8)
        b_blur = masked_blur_score(img_bgr, mask_u8)

        rec = {
            "base_key": base_key,
            "lettuce_id": lettuce_id,
            "slot_idx": int(slot_idx),
            "inst_idx": int(inst_idx),
            "conf": float(conf),
            "shape": [int(H), int(W)],
            "rle": mask_to_rle(mask_u8),
        }
        mask_writer.write(json.dumps(rec, ensure_ascii=False) + "\n")

        rows.append({
            "image_path": img_path,
            "base_key": base_key,
            "lettuce_id": lettuce_id,
            "bed_date": base_key,
            "n_instances": n_instances,
            "conf": float(conf),
            "brightness_mean": float(b_mean),
            "blur_score": float(b_blur),
            "area_px": area_px,
            "area_cm2": area_cm2,
            "px_per_mm_x": pxmm_x,
            "px_per_mm_y": pxmm_y,
            "mm_per_px": mm_per_px,
            "cyl_ok": cyl_ok,
            "cyl_diam_px": cyl_diam_px,
            "front_height_cm": front_height_cm,
            "area_front": area_front,
            "aspect_ratio": aspect_ratio,
            "bottom_flatness": bottom_flat,
            "core_prominence": core_prom,
            "bbox_w": bbox_w,
            "bbox_h": bbox_h,
            "perimeter_px": shape.get("perimeter_px", np.nan),
            "circularity": shape.get("circularity", np.nan),
            "solidity": shape.get("solidity", np.nan),
            "concavity": shape.get("concavity", np.nan),
            "curvature": shape.get("curvature", np.nan),
            "roughness": shape.get("roughness", np.nan),
            "best_instance": int(inst_idx),
            "position_group": position_group,
            "bed_date_clean": bed_date_clean,
            "date": date,
            "px_per_mm": px_per_mm,
        })

    return rows


# =========================
# 7) df 컬럼 정리 (너 df_b.columns 맞추기)
# =========================

REQUIRED_COLS = [
    "image_path", "base_key", "lettuce_id", "bed_date", "n_instances",
    "conf", "brightness_mean", "blur_score", "area_px", "area_cm2",
    "px_per_mm_x", "px_per_mm_y", "mm_per_px", "cyl_ok", "cyl_diam_px",
    "front_height_cm", "area_front", "aspect_ratio", "bottom_flatness",
    "core_prominence", "bbox_w", "bbox_h", "perimeter_px", "circularity",
    "solidity", "concavity", "curvature", "roughness", "best_instance",
    "position_group", "bed_date_clean", "date", "px_per_mm",
]


def finalize_columns(df: pd.DataFrame):
    for c in REQUIRED_COLS:
        if c not in df.columns:
            df[c] = np.nan
    return df[REQUIRED_COLS]



In [7]:

# =========================
# 8) main
# =========================

def main():
    print(f"[{now_hms()}] [INFO] 시작")

    os.makedirs(OUT_DIR, exist_ok=True)
    out_xlsx = os.path.join(OUT_DIR, "features_all_lettuce.xlsx")
    out_mask_jsonl = os.path.join(OUT_DIR, "masks_rle.jsonl")

    # 입력
    paths = collect_image_paths(BED_WARP_FOLDER)
    print(f"[INFO] 베드 warp 이미지 개수: {len(paths)}")
    if len(paths) == 0:
        raise RuntimeError("BED_WARP_FOLDER에 이미지가 없습니다.")

    # scale
    scale_map = load_scale_map(SCALE_CSV)

    # model
    model = YOLO(SEG_MODEL_PATH)
    print("[INFO] model task:", getattr(model, "task", "unknown"))

    # 슬롯(첫 이미지 기준 고정)
    test_img = cv2.imread(paths[0])
    if test_img is None:
        raise RuntimeError("첫 이미지 로딩 실패")
    H0, W0 = test_img.shape[:2]
    slot_names, slot_centers = build_slot_centers(W0, H0, cols=GRID_COLS, rows=GRID_ROWS, x_margin_frac=X_MARGIN_FRAC, y_margin_frac=Y_MARGIN_FRAC)
    max_assign_dist = float(W0 * MAX_ASSIGN_DIST_FRAC)

    print(f"[INFO] 슬롯: {GRID_ROWS}x{GRID_COLS} = {len(slot_names)}개 (t1~t6, b1~b6)")
    print(f"[INFO] max_assign_dist(px): {max_assign_dist:.1f}")

    # 재시작 모드
    rows_all = []
    done_set = set()
    if os.path.exists(out_xlsx):
        try:
            prev = pd.read_excel(out_xlsx)
            if "base_key" in prev.columns:
                rows_all = prev.to_dict(orient="records")
                done_set = set(prev["base_key"].dropna().astype(str).unique().tolist())
                print(f"[INFO] 기존 결과 발견: done base_key={len(done_set)}개 → 재시작")
        except Exception:
            pass

    mask_mode = "a" if os.path.exists(out_mask_jsonl) else "w"

    t0 = time.time()
    n_chunks = (len(paths) - 1) // CHUNK_SIZE + 1

    with open(out_mask_jsonl, mask_mode, encoding="utf-8") as mw:
        for c_start in range(0, len(paths), CHUNK_SIZE):
            chunk = paths[c_start : c_start + CHUNK_SIZE]
            chunk_idx = c_start // CHUNK_SIZE + 1

            chunk2 = [p for p in chunk if parse_base_key_from_path(p) not in done_set]
            if not chunk2:
                continue

            pbar = tqdm(range(0, len(chunk2), BATCH_SIZE), desc=f"Chunk {chunk_idx}/{n_chunks}")

            for i in pbar:
                batch_paths = chunk2[i : i + BATCH_SIZE]

                # 병렬 로딩
                with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
                    imgs = list(ex.map(cv2.imread, batch_paths))

                valid = [(p, im) for p, im in zip(batch_paths, imgs) if im is not None]
                if not valid:
                    continue

                v_paths, v_imgs = zip(*valid)
                results = model(list(v_imgs), verbose=False, conf=PRED_CONF, iou=PRED_IOU)

                elapsed = time.time() - t0
                done = c_start + i + len(batch_paths)
                frac = min(1.0, done / max(1, len(paths)))
                if frac > 0:
                    eta = elapsed * (1 - frac) / frac
                    pbar.set_postfix_str(f"elapsed={elapsed/60:.1f}m eta={eta/60:.1f}m")

                for img_path, img, result in zip(v_paths, v_imgs, results):
                    base_key = parse_base_key_from_path(img_path)
                    if base_key in done_set:
                        continue

                    # 크기가 섞이면 슬롯을 이미지마다 재생성(필요 시)
                    H, W = img.shape[:2]
                    if (H, W) != (H0, W0):
                        _slot_names, _slot_centers = build_slot_centers(W, H, cols=GRID_COLS, rows=GRID_ROWS, x_margin_frac=X_MARGIN_FRAC, y_margin_frac=Y_MARGIN_FRAC)
                        _max_dist = float(W * MAX_ASSIGN_DIST_FRAC)
                    else:
                        _slot_names, _slot_centers, _max_dist = slot_names, slot_centers, max_assign_dist

                    rows = process_bed_image_with_result(img_path, img, result, scale_map, mw, _slot_names, _slot_centers, _max_dist)
                    rows_all.extend(rows)
                    done_set.add(base_key)

            # chunk 중간 저장
            df = finalize_columns(pd.DataFrame(rows_all))
            df.to_excel(out_xlsx, index=False, engine="openpyxl")

    elapsed = time.time() - t0
    print(f"[{now_hms()}] [DONE] saved: {out_xlsx}")
    print(f"[INFO] masks: {out_mask_jsonl}")
    print(f"[INFO] elapsed: {elapsed/60:.1f} min")



In [8]:

if __name__ == "__main__":
    main()


[07:00:20] [INFO] 시작
[INFO] 베드 warp 이미지 개수: 2192
[INFO] scale_map 로드 완료: 2192 rows
[INFO] model task: segment
[INFO] 슬롯: 2x6 = 12개 (t1~t6, b1~b6)
[INFO] max_assign_dist(px): 360.0


Chunk 3/3: 100%|██████████| 16/16 [01:25<00:00,  5.34s/it, elapsed=16.2m eta=0.0m]


[07:16:53] [DONE] saved: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step2. lettuce/features_all_lettuce.xlsx
[INFO] masks: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step2. lettuce/masks_rle.jsonl
[INFO] elapsed: 16.4 min


#이미지의 저장

In [9]:
# ---- 설치(Colab 1회) ----
!pip -q install openpyxl tqdm

In [17]:
# ==============================
# 캔버스 B (실행용)
# Canvas A 산출물(features.xlsx + mask)을 이용해서
# 1) 베드이미지당 t1,t2,b1,b2 상추만 crop 저장
# 2) 베드 단위 2x2 시각화 이미지(옵션) 저장
# ※ YOLO-seg 재실행 없음 (mask 재사용)
# ==============================

import os, json, time
from glob import glob
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

# =========================
# 0) CFG (여기만 수정)
# =========================
FEATURES_XLSX = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step2. lettuce/features_all_lettuce.xlsx"   # Canvas A 결과
# mask 저장 형태 둘 중 하나만 존재해도 됨
MASK_JSONL    = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step2. lettuce/masks_rle.jsonl"             # (선택) jsonl
OUT_DIR       = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step2. lettuce/images"                            # 출력
TARGET_IDS    = ["t1", "t2", "b1", "b2"]              # 베드당 저장할 4개

# crop 옵션
PAD_PX        = 12          # bbox padding
MIN_BOX       = 20          # 너무 작은 bbox는 스킵
MASK_BG       = "black"    # "black" | "white" | "keep" (keep=원본 그대로 crop)
SAVE_EXT      = ".png"      # png 추천

# 실행 옵션
N_WORKERS     = 8
CHUNK_SIZE    = 2000
MAKE_GRID_VIS = True        # bed별 2x2 미리보기 저장
GRID_TILE     = (320, 320)  # 각 타일 resize (w,h)


In [18]:
# =========================
# 1) 유틸
# =========================
def now_hms():
    return time.strftime("%H:%M:%S")

def safe_mkdir(p):
    os.makedirs(p, exist_ok=True)

def rle_to_mask(rle, shape_hw):
    """Canvas A의 mask_to_rle (Fortran order) 역변환"""
    H, W = int(shape_hw[0]), int(shape_hw[1])
    runs = np.array(rle, dtype=np.int64)
    starts = runs[::2] - 1
    lengths = runs[1::2]
    ends = starts + lengths

    flat = np.zeros(H * W, dtype=np.uint8)
    for s, e in zip(starts, ends):
        if e > s:
            flat[s:e] = 1

    mask = flat.reshape((H, W), order='F')
    return mask.astype(np.uint8) * 255


def load_mask_from_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        d = json.load(f)
    # 허용 키: {rle, shape} 또는 {rle, shape:[H,W]} 또는 {shape_hw}
    rle = d.get('rle')
    shape = d.get('shape') or d.get('shape_hw')
    if rle is None or shape is None:
        raise ValueError(f"mask json missing keys: {path}")
    return rle_to_mask(rle, shape)


def build_mask_index_from_jsonl(jsonl_path):
    """(base_key, lettuce_id) -> (rle, shape)"""
    idx = {}
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            d = json.loads(line)
            base_key = str(d.get('base_key'))
            lettuce_id = str(d.get('lettuce_id'))
            rle = d.get('rle')
            shape = d.get('shape')
            if base_key and lettuce_id and rle is not None and shape is not None:
                idx[(base_key, lettuce_id)] = (rle, shape)
    return idx


def get_bbox_from_mask(mask_u8):
    ys, xs = np.where(mask_u8 > 0)
    if xs.size == 0:
        return None
    x1, x2 = int(xs.min()), int(xs.max())
    y1, y2 = int(ys.min()), int(ys.max())
    return x1, y1, x2, y2


def apply_mask_bg(crop_bgr, crop_mask_u8, mode="black"):
    if mode == "keep":
        return crop_bgr

    bg_val = 0 if mode == "black" else 255
    out = np.full_like(crop_bgr, bg_val)
    m = (crop_mask_u8 > 0)
    out[m] = crop_bgr[m]
    return out


def resize_keep_aspect(img, size_wh):
    tw, th = int(size_wh[0]), int(size_wh[1])
    h, w = img.shape[:2]
    if h == 0 or w == 0:
        return np.zeros((th, tw, 3), dtype=np.uint8)
    scale = min(tw / w, th / h)
    nw, nh = max(1, int(w * scale)), max(1, int(h * scale))
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((th, tw, 3), dtype=np.uint8)
    y0 = (th - nh) // 2
    x0 = (tw - nw) // 2
    canvas[y0:y0+nh, x0:x0+nw] = resized
    return canvas

In [19]:
# =========================
# 2) mask provider (jsonl 또는 폴더)
# =========================
MASK_INDEX = None
USE_JSONL = False


def init_mask_provider():
    global MASK_INDEX, USE_JSONL

    if isinstance(MASK_JSONL, str) and os.path.exists(MASK_JSONL):
        print(f"[INFO] mask source: jsonl = {MASK_JSONL}")
        USE_JSONL = True
        # jsonl이 너무 크면 인덱싱을 생략하고, 필요할 때만 scan하는 방식으로 바꿀 수 있음.
        MASK_INDEX = build_mask_index_from_jsonl(MASK_JSONL)
        print(f"[INFO] jsonl index size: {len(MASK_INDEX)}")
        return

    raise RuntimeError("MASK_JSONL 또는 MASK_DIR 중 하나가 유효해야 합니다.")


def get_mask(base_key, lettuce_id):
    if USE_JSONL:
        key = (str(base_key), str(lettuce_id))
        if key not in MASK_INDEX:
            return None
        rle, shape = MASK_INDEX[key]
        return rle_to_mask(rle, shape)

    # dir mode: {base_key}_{lettuce_id}.json
    jpath = os.path.join(MASK_DIR, f"{base_key}_{lettuce_id}.json")
    if not os.path.exists(jpath):
        return None
    return load_mask_from_json(jpath)


# =========================
# 3) 단일 (base_key, lettuce_id) crop 저장
# =========================

def save_one_crop(row, out_crop_dir):
    """return: (base_key, lettuce_id, out_path, ok)"""
    img_path = row['image_path']
    base_key = row['base_key']
    lettuce_id = row['lettuce_id']

    bed_dir = os.path.join(out_crop_dir, base_key)
    safe_mkdir(bed_dir)

    out_name = f"{lettuce_id}{SAVE_EXT}"
    out_path = os.path.join(bed_dir, out_name)


    if os.path.exists(out_path):
        return (base_key, lettuce_id, out_path, True)

    img = cv2.imread(img_path)
    if img is None:
        return (base_key, lettuce_id, out_path, False)

    mask = get_mask(base_key, lettuce_id)
    if mask is None:
        return (base_key, lettuce_id, out_path, False)

    # mask shape != img shape이면 resize (안전)
    if mask.shape[0] != img.shape[0] or mask.shape[1] != img.shape[1]:
        mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)

    bb = get_bbox_from_mask(mask)
    if bb is None:
        return (base_key, lettuce_id, out_path, False)

    x1, y1, x2, y2 = bb
    x1 = max(0, x1 - PAD_PX)
    y1 = max(0, y1 - PAD_PX)
    x2 = min(img.shape[1] - 1, x2 + PAD_PX)
    y2 = min(img.shape[0] - 1, y2 + PAD_PX)

    if (x2 - x1 + 1) < MIN_BOX or (y2 - y1 + 1) < MIN_BOX:
        return (base_key, lettuce_id, out_path, False)

    crop = img[y1:y2+1, x1:x2+1].copy()
    crop_m = mask[y1:y2+1, x1:x2+1].copy()

    crop2 = apply_mask_bg(crop, crop_m, mode=MASK_BG)

    safe_mkdir(out_crop_dir)
    cv2.imwrite(out_path, crop2)

    return (base_key, lettuce_id, out_path, True)

In [20]:
# =========================
# 4) bed별 2x2 시각화
# =========================

def make_bed_grid(base_key, crop_dir, out_vis_dir):
    """t1 t2 / b1 b2 순서 2x2"""
    paths = {lid: os.path.join(crop_dir, f"{base_key}_{lid}{SAVE_EXT}") for lid in TARGET_IDS}
    if not all(os.path.exists(paths[lid]) for lid in TARGET_IDS):
        return None

    imgs = [cv2.imread(paths[lid]) for lid in TARGET_IDS]
    if any(im is None for im in imgs):
        return None

    tiles = [resize_keep_aspect(im, GRID_TILE) for im in imgs]

    top = np.concatenate([tiles[0], tiles[1]], axis=1)  # t1, t2
    bot = np.concatenate([tiles[2], tiles[3]], axis=1)  # b1, b2
    grid = np.concatenate([top, bot], axis=0)

    safe_mkdir(out_vis_dir)
    out_path = os.path.join(out_vis_dir, f"{base_key}_grid.png")
    cv2.imwrite(out_path, grid)
    return out_path


# =========================
# 5) main
# =========================

def main():
    print(f"[{now_hms()}] [INFO] start")

    safe_mkdir(OUT_DIR)
    crop_dir = os.path.join(OUT_DIR, "crops_4")
    vis_dir  = os.path.join(OUT_DIR, "bed_grid_vis")
    safe_mkdir(crop_dir)
    if MAKE_GRID_VIS:
        safe_mkdir(vis_dir)

    if not os.path.exists(FEATURES_XLSX):
        raise RuntimeError("FEATURES_XLSX not found")

    df = pd.read_excel(FEATURES_XLSX)

    # 필수 컬럼 체크
    need = {'image_path', 'base_key', 'lettuce_id'}
    if not need.issubset(set(df.columns)):
        raise ValueError(f"features에 필수 컬럼이 없습니다: need={need}, got={df.columns}")

    # 대상 4개만
    df4 = df[df['lettuce_id'].isin(TARGET_IDS)].copy()
    df4 = df4.dropna(subset=['image_path', 'base_key', 'lettuce_id'])

    print(f"[INFO] 대상 row 수: {len(df4)} (ids={TARGET_IDS})")

    # mask provider 준비
    init_mask_provider()

    # 재시작: 이미 저장된 crop는 자동 스킵(save_one_crop에서 체크)
    rows = df4.to_dict(orient='records')

    t0 = time.time()
    n_chunks = (len(rows) - 1) // CHUNK_SIZE + 1

    ok_count = 0
    fail_count = 0

    for c_start in range(0, len(rows), CHUNK_SIZE):
        chunk = rows[c_start:c_start + CHUNK_SIZE]
        chunk_idx = c_start // CHUNK_SIZE + 1

        pbar = tqdm(chunk, desc=f"Chunk {chunk_idx}/{n_chunks}")

        # 병렬 crop
        with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
            for base_key, lettuce_id, out_path, ok in ex.map(lambda r: save_one_crop(r, crop_dir), pbar):
                if ok:
                    ok_count += 1
                else:
                    fail_count += 1

        elapsed = time.time() - t0
        done = min(len(rows), c_start + len(chunk))
        frac = done / max(1, len(rows))
        eta = elapsed * (1 - frac) / max(frac, 1e-9)
        print(f"[{now_hms()}] chunk done {done}/{len(rows)} | ok={ok_count} fail={fail_count} | elapsed={elapsed/60:.1f}m eta={eta/60:.1f}m")

    # bed grid
    if MAKE_GRID_VIS:
        bases = sorted(df4['base_key'].astype(str).unique().tolist())
        print(f"[INFO] make grid: base_key={len(bases)}")

        # grid는 IO라 병렬 가능
        def _grid(bk):
            return make_bed_grid(bk, crop_dir, vis_dir)

        with ThreadPoolExecutor(max_workers=max(2, N_WORKERS//2)) as ex:
            for _ in tqdm(ex.map(_grid, bases), total=len(bases), desc="bed_grid"):
                pass

    print(f"[{now_hms()}] [DONE] crops saved to: {crop_dir}")
    if MAKE_GRID_VIS:
        print(f"[{now_hms()}] [DONE] bed grids saved to: {vis_dir}")



In [21]:

if __name__ == "__main__":
    main()


[07:46:02] [INFO] start
[INFO] 대상 row 수: 4372 (ids=['t1', 't2', 'b1', 'b2'])
[INFO] mask source: jsonl = /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step2. lettuce/masks_rle.jsonl
[INFO] jsonl index size: 20443


Chunk 1/3: 100%|██████████| 2000/2000 [00:00<00:00, 112247.71it/s]


[07:46:35] chunk done 2000/4372 | ok=2000 fail=0 | elapsed=0.4m eta=0.5m


Chunk 2/3: 100%|██████████| 2000/2000 [00:00<00:00, 115191.74it/s]


[07:47:01] chunk done 4000/4372 | ok=4000 fail=0 | elapsed=0.8m eta=0.1m


Chunk 3/3: 100%|██████████| 372/372 [00:00<00:00, 73709.42it/s]


[07:47:06] chunk done 4372/4372 | ok=4372 fail=0 | elapsed=0.9m eta=0.0m
[INFO] make grid: base_key=1993


bed_grid: 100%|██████████| 1993/1993 [00:00<00:00, 5508.59it/s]

[07:47:06] [DONE] crops saved to: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step2. lettuce/images/crops_4
[07:47:06] [DONE] bed grids saved to: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/step2. lettuce/images/bed_grid_vis
